# 13. Gün — Proje Tekrarı ve Bilgi Pekiştirme

Bugün projeye yeni bir model, özellik veya pipeline bileşeni eklemek yerine,
önceki günlerde gerçekleştirilen çalışmaların kapsamlı bir tekrarı yapıldı.

Amaç, geliştirilen sistemin yalnızca kod seviyesinde çalışmasını değil;
kullanılan yöntemlerin neden seçildiğinin, proje dosyalarının görevlerinin
ve veri bilimi sürecinin bütün olarak anlaşılmasının pekiştirilmesiydi.

## Tekrar Edilen Başlıca Konular

### Veri Toplama ve Proje Yapısı

Google Trends verilerinin 0–100 arasında normalize edilmiş göreli ilgi
değerleri olduğu ve gerçek arama hacmini temsil etmediği tekrar edildi.

`pytrends` ile veri çekme süreci, `geo` ve `timeframe` parametrelerinin
sorgunun normalizasyon bağlamını değiştirmesi ve `isPartial` gözlemlerinin
neden modelleme verisinden çıkarıldığı gözden geçirildi.

Veri çekme işlemlerinin tekrar kullanılabilir hale getirilmesi amacıyla
oluşturulan `src/fetch_data.py` dosyasının görevi; type hint, input
validation, exception handling ve rate-limit problemlerinin yönetimi
tekrar edildi.

Ayrıca projenin temel klasör yapısı gözden geçirildi:

- `data/raw/`: ham veriler
- `data/processed/`: temizlenmiş ve modellemeye hazır veriler
- `notebooks/`: analiz ve deney çalışmaları
- `src/`: tekrar kullanılabilir proje kodları
- `models/`: eğitilmiş model ve metadata dosyaları
- `reports/`: final tahmin çıktıları

### Veri Hazırlama ve EDA

ChatGPT, Gemini ve Claude için global üç yıllık haftalık Google Trends
verisinin neden ana veri seti olarak seçildiği tekrar edildi.

Eksik değer kontrolü, zaman indeksinin önemi, veri tipleri, temel
istatistikler ve zaman serisinin davranışının modelleme öncesinde
incelenmesi gibi EDA adımları gözden geçirildi.

### Forecasting ve Model Evaluation

İlk referans model olarak kullanılan Naive Forecast yaklaşımı tekrar edildi.

Zaman serilerinde random train-test split kullanılmaması gerektiği;
gelecek bilginin training verisine sızmasını engellemek için kronolojik
ayrım yapılmasının önemi tekrar değerlendirildi.

Model performansının değerlendirilmesinde:

- MAE
- RMSE

metriklerinin anlamları ve aralarındaki farklar tekrar edildi.

ARIMA modelindeki `p`, `d`, `q`, lag, differencing ve stationarity
kavramları gözden geçirildi.

Tek bir train-test döneminin model performansını değerlendirmek için
yetersiz kalabileceği ve bu nedenle expanding-window Time-Series
Cross-Validation yaklaşımına geçildiği tekrar edildi.

### Prophet

Prophet modelinin:

- `ds` ve `y` veri formatı
- trend
- changepoint
- seasonality
- `changepoint_prior_scale`

kavramları tekrar edildi.

Veri frekansı ile seasonality kavramlarının aynı olmadığı ve haftalık
veride günlük/haftanın günlerine dayalı seasonality bileşenlerinin neden
kullanılmadığı gözden geçirildi.

### XGBoost ve Feature Engineering

Zaman serisinin supervised-learning problemine dönüştürülmesi tekrar edildi.

`shift()` kullanılarak oluşturulan:

- `lag_1`–`lag_8`
- `change_1`
- `change_2`

feature'larının amaçları incelendi.

Dört haftalık tahmin üretmek için kullanılan recursive forecasting
yaklaşımı ve önceki tahminlerin sonraki tahminlerde input olarak
kullanılması nedeniyle oluşabilecek error propagation problemi tekrar edildi.

XGBoost modelindeki:

- `n_estimators`
- `max_depth`
- `learning_rate`
- `random_state`

parametrelerinin görevleri gözden geçirildi.

Prophet ve XGBoost'un farklı modelleme yaklaşımlarından yararlanması
nedeniyle oluşturulan %50-%50 Ensemble modelinin ChatGPT serisinde daha
iyi out-of-sample sonuç verdiği tekrar edildi.

### Teknoloji Bazında Model Seçimi

Her trend serisine aynı modelin zorlanmaması gerektiği tekrar edildi.

Güncel model seçimleri:

- ChatGPT → Prophet + XGBoost Ensemble
- Gemini → Naive
- Claude → Naive

şeklindedir.

Model seçimlerinin sabit olmadığı, yeni veriler geldiğinde Time-Series
Cross-Validation sonuçlarının ve dolayısıyla seçilen modelin
değişebileceği tekrar değerlendirildi.

### Veri Güncelleme Süreci

Yeni Google Trends verileri ana veri setine eklenirken farklı sorgu
pencerelerinin normalize edilmiş ölçeklerinin doğrudan aynı kabul
edilemeyeceği tekrar edildi.

Bir yıllık ve üç yıllık sorguların ortak yaklaşık 51 haftalık döneminde
ölçek karşılaştırması yapılmış, median ratio yaklaşık `1.0` bulunduğu için
ek bir rescaling uygulanmadan yalnızca yeni haftalar ana veri setine
eklenmişti.

Google Trends skorlarının doğal olarak `[0, 100]` aralığında bulunması
gerektiğinden bütün modeller için aynı clipping kuralının uygulanmasının
önemi tekrar edildi.

### Monitoring

`src/monitoring.py` içerisinde kullanılan anomaly detection yaklaşımı
tekrar edildi.

Anomaly hesaplamasında:

- 12 haftalık rolling window
- rolling mean
- rolling standard deviation
- `shift(1)`
- anomaly score
- `threshold = 3.5`
- `min_absolute_change = 5`

kavramları tekrar gözden geçirildi.

`shift(1)` kullanımının mevcut gözlemin kendi referans istatistiklerine
dahil olmasını engellediği ve böylece leakage riskini azalttığı tekrar
edildi.

Event-aware XGBoost deneyi de gözden geçirildi. Event feature'ının
out-of-sample performansta anlamlı iyileşme sağlamaması ve feature
importance değerinin sıfır çıkması nedeniyle final forecasting
pipeline'ına dahil edilmediği hatırlatıldı.

### Model Persistence ve Final Pipeline

Final ChatGPT Prophet ve XGBoost modellerinin native JSON formatında
kaydedilmesi ve tekrar yüklenen modellerin aynı prediction'ları
ürettiğinin doğrulanması tekrar edildi.

Dosyaların görevleri gözden geçirildi:

- `src/forecasting.py`: model training, evaluation ve forecasting
- `src/monitoring.py`: anomaly ve trend sinyalleri
- `src/pipeline.py`: final inference akışı
- `app.py`: Streamlit kullanıcı arayüzü

Model artifact, model metadata ve forecast report kavramlarının
birbirinden farklı görevler taşıdığı tekrar edildi.

### Forecast Uncertainty

Point forecast'ın tek başına gelecek hakkındaki belirsizliği
göstermediği tekrar edildi.

`forecast ± MAE` yerine Time-Series Cross-Validation'dan elde edilen
out-of-sample residual dağılımlarının kullanılması gözden geçirildi.

Residual:

`Actual - Prediction`

olarak tanımlanmıştır.

12 fold ve her fold'da 4 haftalık test kullanılarak her trend için
48 out-of-sample residual elde edilmiştir.

Residual dağılımlarının Q10 ve Q90 quantile değerleri kullanılarak
merkezi yaklaşık %80'lik ampirik prediction interval oluşturulmuştur.

Her horizon için yalnızca 12 residual bulunduğundan daha stabil quantile
hesabı amacıyla residual'lar pooled olarak kullanılmıştır.

Prediction interval'ların historical empirical coverage değerleri ve:

- `Lower <= Forecast <= Upper`
- eksik değer bulunmaması
- bütün değerlerin `[0, 100]` aralığında bulunması

gibi yapısal kontroller tekrar gözden geçirilmiştir.

## Gün Sonu Değerlendirmesi

Bugünkü çalışma yeni bir model veya özellik geliştirmekten ziyade,
projenin başından itibaren alınan teknik kararların ve kullanılan veri
bilimi yöntemlerinin bütünsel olarak tekrar edilmesine ayrıldı.

Bu tekrar sayesinde veri toplama aşamasından model evaluation,
feature engineering, monitoring, model persistence, inference pipeline,
dashboard ve forecast uncertainty aşamalarına kadar projenin uçtan uca
mantığı yeniden gözden geçirilmiş ve proje sunumu/raporlaması öncesinde
teknik bilgi pekiştirilmiştir.